# Business Entity Resolution -- run the pipeline end to end

Runs every stage of the ML Challenge 2026 Business Entity Resolution pipeline, in the order
`code/business_entity_resolution/README.md` documents: normalize -> blocking -> features -> GBDT
-> Laya fine-tuning -> ensemble -> test-set inference -> validate. Every cell shells out to the
actual scripts in `src/` (`!python -m src.<script> ...`) rather than re-implementing anything
inline, so this notebook and the repo can never drift apart -- if you change a script, this
notebook picks it up automatically on the next run.

**Source:** [https://github.com/prathameshfuke/azlaya](https://github.com/prathameshfuke/azlaya), branch `claude/determined-lovelace-17n6zt`.

**Requires a GPU** for the Laya fine-tuning cells (Section 5) -- everything before that (Sections
1-4) runs fine on CPU. On Colab: Runtime -> Change runtime type -> GPU. On Kaggle: Notebook
options -> Accelerator -> GPU T4 x2 (only needed if you want to run the `torchrun` DDP variant in
5c; a single T4/A100 works fine with the default single-process cells).

**Before you run anything**: this pipeline has never been executed end-to-end (it was authored
and only statically checked in a no-GPU environment) -- read each stage's markdown cell and its
printed output as you go, don't just Run All and walk away. Section 3's blocking-recall number in
particular is worth stopping on: if it's low, nothing downstream can fix it.

Run cells **top to bottom, in order** -- later stages read files earlier stages write.


## 0. Setup

### 0.1 Check the GPU


In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found -- no GPU visible yet.")

try:
    import torch
    print(f"torch {torch.__version__} already installed, CUDA available: {torch.cuda.is_available()}")
except ImportError:
    print("torch not installed yet -- that's expected before the next cell installs requirements.txt.")


### 0.2 Clone the repository and install dependencies

Edit `REPO_URL` / `BRANCH` below if you're running from a fork or a merged `main`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/prathameshfuke/azlaya"
BRANCH = "claude/determined-lovelace-17n6zt"

# /kaggle/working on Kaggle, /content on Colab, else the notebook's own directory.
if os.path.isdir("/kaggle/working"):
    WORKDIR = "/kaggle/working"
elif os.path.isdir("/content"):
    WORKDIR = "/content"
else:
    WORKDIR = os.getcwd()

REPO_DIR = f"{WORKDIR}/azlaya"
PIPELINE_DIR = f"{REPO_DIR}/student_resource/code/business_entity_resolution"
STUDENT_RESOURCE_DIR = f"{REPO_DIR}/student_resource"

if not os.path.isdir(REPO_DIR):
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR],
        capture_output=True, text=True,
    )
    print(result.stdout, result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"git clone failed (exit {result.returncode}) -- check REPO_URL/BRANCH above.")
else:
    print(f"{REPO_DIR} already exists -- skipping clone (delete it first if you want a fresh checkout).")

print("PIPELINE_DIR:", PIPELINE_DIR)


In [ ]:
%cd $PIPELINE_DIR
!pip install -q -r requirements.txt
import laya, torch, transformers
print("laya", laya.__version__, "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| CUDA available:", torch.cuda.is_available())


### 0.3 Provide the dataset

`dataset/` isn't tracked in the repo (it's large and gitignored), so it needs to land at
`student_resource/dataset/{train,test}/*.tsv` before anything else can run. Use **whichever one
cell below matches your setup** and skip the others -- all three end with the same layout, and the
verification cell after them checks it regardless of which one you used.


In [ ]:
# Option A -- Google Drive (Colab). Set DRIVE_DATASET_DIR to the folder in your Drive that
# directly contains train/ and test/ subfolders, then run this cell.
RUN_OPTION_A = False
DRIVE_DATASET_DIR = "/content/drive/MyDrive/azlaya-dataset"  # <-- edit this

if RUN_OPTION_A:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(f"{STUDENT_RESOURCE_DIR}/dataset", exist_ok=True)
    os.system(f"cp -r {DRIVE_DATASET_DIR}/train {STUDENT_RESOURCE_DIR}/dataset/")
    os.system(f"cp -r {DRIVE_DATASET_DIR}/test {STUDENT_RESOURCE_DIR}/dataset/")
    print("Copied from Google Drive.")


In [ ]:
# Option B -- Kaggle Dataset attached to this notebook. Set KAGGLE_INPUT_DIR to wherever the
# challenge files landed under /kaggle/input (check the Data pane on the right), then run this.
RUN_OPTION_B = False
KAGGLE_INPUT_DIR = "/kaggle/input/ml-challenge-2026-business-entity-resolution"  # <-- edit this

if RUN_OPTION_B:
    os.makedirs(f"{STUDENT_RESOURCE_DIR}/dataset", exist_ok=True)
    os.system(f"cp -r {KAGGLE_INPUT_DIR}/train {STUDENT_RESOURCE_DIR}/dataset/")
    os.system(f"cp -r {KAGGLE_INPUT_DIR}/test {STUDENT_RESOURCE_DIR}/dataset/")
    print("Copied from Kaggle input.")


In [ ]:
# Option C -- direct upload (Colab). Upload a zip containing train/ and test/ at its root.
RUN_OPTION_C = False

if RUN_OPTION_C:
    from google.colab import files
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    os.makedirs(f"{STUDENT_RESOURCE_DIR}/dataset", exist_ok=True)
    os.system(f"unzip -q -o {zip_name} -d {STUDENT_RESOURCE_DIR}/dataset")
    print("Unzipped upload.")


In [ ]:
# Verify the dataset landed where every script expects it, regardless of which option you used.
required = [
    "dataset/train/train_source1.tsv", "dataset/train/train_source2.tsv",
    "dataset/train/train_source3.tsv", "dataset/train/train_ground_truth.tsv",
    "dataset/test/test_source1.tsv", "dataset/test/test_source2.tsv", "dataset/test/test_source3.tsv",
]
missing = [f for f in required if not os.path.isfile(f"{STUDENT_RESOURCE_DIR}/{f}")]
if missing:
    raise FileNotFoundError(
        "Missing dataset files, run one of the Option A/B/C cells above first:\n  " + "\n  ".join(missing)
    )
print("All six source files + ground truth found under", f"{STUDENT_RESOURCE_DIR}/dataset")


## 1. Normalize

Cleans `business_name`/`business_address` on all six source files, expands country-aware legal
suffixes, splits DBA names, extracts pin/postal code + a city guess, and tags each name/address
field's script (Latin / Devanagari / Tamil / other). See `src/normalize.py` for the full list of
what this does and does not attempt (the module docstring flags the specific heuristics worth
spot-checking).


In [ ]:
!python -m src.normalize --split train --split test


In [ ]:
# Spot-check a handful of normalized rows before trusting the rest of the pipeline to them.
import pandas as pd
sample = pd.read_csv(f"{STUDENT_RESOURCE_DIR}/data_processed/train_source1_normalized.tsv", sep="\t", dtype=str).head(10)
sample[["business_name", "name_norm", "has_dba", "trade_name_norm", "business_address", "address_norm", "pin_code", "city_guess", "name_script"]]


## 2. Blocking (candidate generation)

Generates up to K=30 S2/S3 candidates per S1 entity, partitioned by country (an open set --
nothing here is hardcoded to {US, India}) via a name-token index, a character-trigram index, and
a pin/city index. `--report-recall` also prints the blocking-recall ceiling on a held-out val
split -- **read that number before continuing**; it upper-bounds everything downstream.


In [ ]:
!python -m src.blocking --split train --k 30 \
    --out ../../data_processed/candidate_pairs_train.tsv --report-recall


If `macro_entity_recall` above looks low, try a larger `--k` or `--score-method tfidf` and re-run
this cell before moving on -- it's much cheaper to fix here than after training the GBDT/Laya.


## 3. Feature engineering

Computes Levenshtein, Jaro-Winkler, word/char TF-IDF cosine, pin/city exact match, country match,
and DBA-aware features for every (S1, candidate) pair from the previous step.


In [ ]:
!python -m src.features --split train --candidates ../../data_processed/candidate_pairs_train.tsv


In [ ]:
feat = pd.read_parquet(f"{STUDENT_RESOURCE_DIR}/data_processed/features_train.parquet")
print(feat.shape)
feat.head()


## 4. GBDT matcher

Trains a LightGBM binary classifier on the feature table (stratified by whole S1 entity, not by
row, so no validation entity's pairs leak into training) and reports precision/recall/macro F_0.5
on the held-out split.


In [ ]:
!python -m src.train_gbdt --features ../../data_processed/features_train.parquet


In [ ]:
import json
report = json.load(open(f"{STUDENT_RESOURCE_DIR}/data_processed/gbdt_val_report.json"))
print(json.dumps(report, indent=2))


## 5. Laya fine-tuning

Two checkpoints, fine-tuned separately, following Laya's own RLCD recipe
(`src/laya_finetune.py`): `english` on Latin-script pairs only, `multilingual` on every pair
(so it also covers India's Devanagari/Tamil business names). This is the GPU-heavy part of the
notebook -- Section 0.1's GPU check should show CUDA available before you run 5b/5c.

### 5a. Build the fine-tuning datasets (CPU, fast, no GPU/laya import needed)


In [ ]:
!python -m src.laya_finetune --stage prepare


In [ ]:
# Peek at a couple of built examples before spending GPU time on the rest.
import json as _json
with open(f"{STUDENT_RESOURCE_DIR}/data_processed/laya_english_train.jsonl") as f:
    for i, line in zip(range(2), f):
        print(_json.dumps(_json.loads(line), indent=2))
        print("---")


### 5b. Fine-tune the English checkpoint

Single process -- works on any single GPU (Colab's default runtime included).


In [ ]:
!python -m src.laya_finetune --stage train --role english


### 5c. Fine-tune the multilingual checkpoint

Use the single-process cell on a single-GPU runtime (Colab). If you're on Kaggle with the
**GPU T4 x2** accelerator selected, use the `torchrun` cell instead for real DDP across both T4s
-- run only one of the two.


In [ ]:
# Single GPU (Colab, or Kaggle with only one GPU selected)
!python -m src.laya_finetune --stage train --role multilingual


In [ ]:
# Kaggle 2xT4 DDP alternative -- run this INSTEAD of the cell above, not in addition to it.
RUN_DDP_MULTILINGUAL = False
if RUN_DDP_MULTILINGUAL:
    !torchrun --standalone --nproc_per_node=2 -m src.laya_finetune --stage train --role multilingual


Both fine-tuned checkpoints now live under `models/laya_english/` and `models/laya_multilingual/`
(plus each stage's `rl_agent_config.json` with its fitted calibration temperature).


## 6. Ensemble

Scores every candidate pair with the fine-tuned Laya `Router`, stacks `laya_prob` onto the GBDT's
feature table alongside `gbdt_prob`, retrains a logistic-regression stacker (primary) and a
shallow-GBDT stacker (alternative), and refits the F_0.5 threshold on the ensemble's own output --
broken down by country (France especially, since Laya never trains on it) and by singleton status.


In [ ]:
!python -m src.ensemble --features ../../data_processed/features_train.parquet


In [ ]:
ens_report = json.load(open(f"{STUDENT_RESOURCE_DIR}/data_processed/ensemble_val_report.json"))
for name in ("logistic", "gbdt_alt"):
    best = ens_report[name]["threshold_sweep"]["best"]
    print(f"{name}: threshold={best['threshold']} macro_F0.5={best['macro_f0_5']:.4f} "
          f"precision={best['precision']:.4f} recall={best['recall']:.4f}")

BEST_STACK_MODEL = max(("logistic", "gbdt_alt"), key=lambda n: ens_report[n]["threshold_sweep"]["best"]["macro_f0_5"])
print("\nBest stack model on this val split:", BEST_STACK_MODEL)
print("\nCompare against Section 4's GBDT-only report above -- only proceed with Laya in the loop",
      "if this genuinely beats it; otherwise --stack-model alone won't save a broken ensemble.")

print(json.dumps(ens_report[BEST_STACK_MODEL]["breakdown"], indent=2))


## 7. Full test-set inference

Runs blocking -> features -> GBDT -> Laya -> ensemble -> threshold on the TEST set (uses
`BEST_STACK_MODEL` from Section 6), producing the two submission files.


In [ ]:
!python -m src.predict --stack-model $BEST_STACK_MODEL


## 8. Validate before submitting

Run from `student_resource/`, exactly as the challenge's own validator expects.


In [ ]:
%cd $STUDENT_RESOURCE_DIR
!python3 utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir dataset/test
%cd $PIPELINE_DIR


## 9. (Optional) Package the submission zip

Bundles `output/`, `code/business_entity_resolution/`, and `Documentation_template.md` into the
zip structure the challenge asks for, and downloads it if running on Colab.


In [ ]:
import shutil

zip_root = f"{WORKDIR}/submission_package"
if os.path.isdir(zip_root):
    shutil.rmtree(zip_root)
os.makedirs(zip_root)
shutil.copytree(f"{STUDENT_RESOURCE_DIR}/output", f"{zip_root}/output")
shutil.copytree(PIPELINE_DIR, f"{zip_root}/code/business_entity_resolution")
shutil.copy(f"{STUDENT_RESOURCE_DIR}/Documentation_template.md", f"{zip_root}/Documentation_template.md")

archive_path = shutil.make_archive(f"{WORKDIR}/submission", "zip", zip_root)
print("Wrote", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Not on Colab -- find the zip at", archive_path, "(on Kaggle: Output pane after committing the notebook).")
